In [1]:
import geopandas as gpd
from pathlib import Path

In [2]:
address_gdf = gpd.read_parquet("../../data/processed/nsw_addressing/addresspoint_all.parquet")
flood_gdf = gpd.read_parquet("../../data/processed/nsw_flood/flood.parquet")

In [3]:
print("Address rows:", len(address_gdf))
print("Flood rows:", len(flood_gdf))

print("Address CRS:", address_gdf.crs)
print("Flood CRS:", flood_gdf.crs)

print(address_gdf.columns)
print(flood_gdf.columns)

Address rows: 4225113
Flood rows: 622
Address CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction"

In [4]:
flood_keep = flood_gdf[
    [
        "OBJECTID",
        "EPI_NAME",
        "LGA_NAME",
        "LAY_CLASS",
        "EPI_TYPE",
        "COMMENT",
        "geometry",
    ]
].copy()

if address_gdf.crs != flood_keep.crs:
    address_gdf = address_gdf.to_crs(flood_keep.crs)

sample_address = address_gdf.sample(50000, random_state=42).copy()
print(len(sample_address))

50000


In [5]:
joined = gpd.sjoin(
    sample_address,
    flood_keep,
    how="left",
    predicate="within"
)

print("Joined rows:", len(joined))
joined[
    ["address", "LAY_CLASS", "COMMENT", "EPI_NAME", "LGA_NAME"]
].head(10)

Joined rows: 50141


,address,LAY_CLASS,COMMENT,EPI_NAME,LGA_NAME
2262872,GRINTON AVENUE ASHMONT,NaN,NaN,NaN,NaN
136524,58 VISTA AVENUE CATALINA,NaN,NaN,NaN,NaN
3400623,5/40 TRAIN STREET BROULEE,NaN,NaN,NaN,NaN
975625,13 BACH AVENUE EMERTON,NaN,NaN,NaN,NaN
1229084,12 GUILFOYLE PLACE CUDGEN,NaN,NaN,NaN,NaN
3506896,14 SPURFIELD ROAD MCLEANS RIDGES,NaN,NaN,NaN,NaN
2094798,1/43 THE TRONGATE GRANVILLE,NaN,NaN,NaN,NaN
1372500,39 MANNS ROAD NARARA,NaN,NaN,NaN,NaN
466580,4 CAROLINA CRESCENT MUDGEE,NaN,NaN,NaN,NaN
3991008,62 WADE STREET COOLAMON,NaN,NaN,NaN,NaN


In [9]:
joined["LAY_CLASS"].isna().mean()

np.float64(0.9907062084920524)

In [10]:
flood_hits = joined[joined["LAY_CLASS"].notna()].copy()
print("Flood hits:", len(flood_hits))

flood_hits[
    ["address", "LAY_CLASS", "COMMENT", "EPI_NAME", "LGA_NAME"]
].sample(min(20, len(flood_hits)), random_state=42)

Flood hits: 466


,address,LAY_CLASS,COMMENT,EPI_NAME,LGA_NAME
481011,142 MARY STREET GRAFTON,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
2745925,1/15 UKI STREET YAMBA,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
1228549,4 GUNDAROO CRESCENT ILUKA,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
562602,3 TIARA CLOSE GRAFTON,Flood Planning Area,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
4086994,160/90 CARRS DRIVE YAMBA,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
8963,2/40 BAYVIEW DRIVE YAMBA,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
1014042,33 BRUCE STREET GRAFTON,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
562486,232 HOOF STREET GRAFTON,Probable Maximum Flood Line,None,Clarence Valley Local Environmental Plan 2011,CLARENCE VALLEY
2033069,198 LOWER BATHURST STREET FORBES,Flood Planning Area,None,Forbes Local Environmental Plan 2013,FORBES
3928610,67 MCCLYMONT DRIVE CATHERINE FIELD,Flood Prone and Major Creeks Land,None,State Environmental Planning Policy (Precincts...,


In [11]:
joined["rid"].duplicated().mean()

np.float64(0.0028120699627051716)

In [12]:
joined["flood_flag"] = joined["LAY_CLASS"].notna().astype(int)
joined["flood_flag"].value_counts(normalize=True)

flood_flag
0    0.990706
1    0.009294
Name: proportion, dtype: float64